# Creating Ensemble Model

VotingRegressor pretrained modelleri desteklemedi?i i?in terkardan kendimiz e?itmek zorunday?z.

## Setup Enviroment

In [ ]:
import joblib
import re
import optuna
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
from lightgbm import LGBMRegressor
import lightgbm as lgb
from sklearn.model_selection import cross_validate
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.exceptions import ConvergenceWarning
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import train_test_split, cross_val_score,GridSearchCV,cross_validate

In [ ]:
from sklearn.ensemble import VotingRegressor
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import make_scorer

In [ ]:
hgbr_model = joblib.load("hgbr.pkl")

In [ ]:
lgbmr_model = joblib.load("lgbm_model.pkl")

## Dataset Process...

### Loading Datasets

In [ ]:
df2_ = pd.read_excel("kaggle_test2.xlsx",index_col="Unnamed: 0")
df_ = pd.read_csv("dataset.csv", encoding="unicode-escape", sep=",",on_bad_lines="skip",index_col="Unnamed: 0")

In [ ]:
df = df_.copy()

### Preprocess

In [ ]:
# De?i?ken isimlerini k???lt?yorum
df.columns = [col.lower() for col in df.columns]

# Ba??ml? de?i?kenin t?revlerini overfit olmamas? i?in ??kart?yorum.
df.drop(["md","ml","mw","ms","mb"],axis=1,inplace=True)

# Date de?i?keninin olu?turulmas?
df["olus tarihi"] = df["olus tarihi"].astype(str)
df["olus zamani"] = df["olus zamani"].astype(str)
# tip de?i?keni 2 s?n?f ve SM s?n?f?ndan sadece 2 g?zlem var. O y?zden dropluyorum bu kolonu.
df["depremin_olus_zamani"] = df["olus tarihi"] + " " + df["olus zamani"]
df.drop(["olus tarihi", "olus zamani", "tip"], axis = 1, inplace = True)
df["depremin_olus_zamani"] = pd.to_datetime(df["depremin_olus_zamani"])

# ?o?ullanm?? ID'leri d???r?yorum.
df.drop_duplicates(subset=["deprem kodu"],inplace = True)

# T?rk?e karakterleri ?ngilizce karakterlere d?n??t?ren bir ?eviri s?zl??? olu?tural?m
turkce_ingilizce_ceviri = {
    '?': 'c',
    '?': 'g',
    '?': 'i',
    '?': 'o',
    '?': 's',
    '?': 'u',
    '?': 'C',
    '?': 'G',
    '?': 'I',
    '?': 'O',
    '?': 'S',
    '?': 'U'
}
df["yer"] = df["yer"].apply(lambda yer: yer.translate(str.maketrans(turkce_ingilizce_ceviri)))

# ?ehir isimlerini ?ekmek i?in d?zenli ifade kullanal?m

def extract_cities(dataframe,column):
  sadece_sehirler = []
  for yer in dataframe[column]:
      eslesme = re.search(r'\((.*?)\)', yer)  # Parantez i?indekileri kontrol et
      if eslesme:
          sadece_sehirler.append(eslesme.group(1))  # Parantez i?indeki de?eri ekle
      else:
          sadece_sehirler.append(yer)  # Parantez i?inde de?er yoksa do?rudan ekle
  return sadece_sehirler

df["temizlenmis_yer"] = extract_cities(df,"yer")

# K??eli parantezleri at?yoruz. B?ylelikle o ?ehir/alandaki depremlere odaklanm?? oluyoruz.
df["temizlenmis_yer"] = df["temizlenmis_yer"].apply(lambda yer: re.sub(r'\[.*?\]', '', yer))

# Yerin neresi oldu?u belli olmad??? i?in ??kart?yorum.
df = df[~(df["temizlenmis_yer"] == "#NAME?")]

In [ ]:
df.temizlenmis_yer.nunique()

### Joining Two Tables

In [ ]:
df = pd.concat([df,df2_],axis=0,ignore_index=True)
df.head()

In [ ]:
df.temizlenmis_yer.nunique()

## Feature Engineering

### Creating DateTime

In [ ]:
def creating_datetime_feature(dataframe):
    dataframe["ay"] = dataframe["depremin_olus_zamani"].dt.month
    dataframe["yil"] = dataframe["depremin_olus_zamani"].dt.year
    dataframe["ay_gun"] = dataframe["depremin_olus_zamani"].dt.day
    dataframe["hafta_gun"] = dataframe["depremin_olus_zamani"].dt.dayofweek
    dataframe["saat"] =  dataframe["depremin_olus_zamani"].dt.hour
    dataframe["aksam"] = ((dataframe['saat'] >= 18) & (dataframe['saat'] <= 23)) | ((dataframe['saat'] >= 0) & (dataframe['saat'] <= 4))
    dataframe['y?l_g?n'] = dataframe["depremin_olus_zamani"].dt.dayofyear
    dataframe['y?l_hafta'] = dataframe["depremin_olus_zamani"].dt.weekofyear
    dataframe["ay_basi"] = dataframe["depremin_olus_zamani"].dt.is_month_start.astype(int)
    dataframe["ay_sonu"] = dataframe["depremin_olus_zamani"].dt.is_month_end.astype(int)
    dataframe["?eyreklik"] = dataframe["depremin_olus_zamani"].dt.quarter
    return dataframe
df = creating_datetime_feature(df)

In [ ]:
# Ak?am de?i?keni Integer tipine d?n??t?r?l?yor. 
df["aksam"] = df['aksam'].astype(int)

### Lag Feature

In [ ]:
# random noise. Ba??ml? de?i?kenden t?retece?imiz featurelarda, overfitin ?n?ne ge?mek i?in g?r?lt? olu?turaca??z.
def random_noise(dataframe):
    return np.random.normal(scale=1.6, size=(len(dataframe)))

In [ ]:
# Zamana g?re s?ral?yoruz Zaman Serilerini.
df = df.sort_values(by = ["depremin_olus_zamani","temizlenmis_yer"],
               axis = 0)

In [ ]:
# lag features. Shift ile gecikmeyi sa?l?yoruz. 
# Random noise ile g?r?lt? ekliyoruz. 
# Transform ile de zaten d?n??t?rme yap?yoruz.
def lag_features(dataframe, lags):
    for lag in lags:
        dataframe['deprem_lag_' + str(lag)] = dataframe.groupby(["temizlenmis_yer"])['xm'].transform(
            lambda x: x.shift(lag)) + random_noise(dataframe)
    return dataframe

In [ ]:
# ?e?itli shiftler/lagler/gecikmeler girelim. Gecikme  o zamanki de?er demekti unutma. 3 ay ve katlar? olacak ?ekilde bak?yorum! Quarter ald?m.
# Gelecekten Not: ?uanl?k veri setinde g?n g?n g?zlem var. Bunlar? ?eyreklik yap?nca SMAPE de?erimiz artt?!
df = lag_features(df, [1, 3, 7, 30, 60, 90, 180, 360])

In [ ]:
df.isnull().sum()

### Rolling Mean Features

In [ ]:
# Rolling Mean Feature
# Window parametresi, ad?m say?s?n? belirtmek i?in kullan?l?yor.
def roll_mean_features(dataframe, windows):
    for window in windows:
        dataframe['deprem_roll_mean_' + str(window)] = dataframe.groupby(["temizlenmis_yer"])['xm']. \
                                                          transform(
            lambda x: x.shift(1).rolling(window=window, min_periods=10, win_type="triang").mean()) + random_noise(
            dataframe)
    return dataframe

In [ ]:
# Min Periodu 10 yapt???m?z i?in min window'a da 10 girebiliyoruz.
df = roll_mean_features(df, [10, 30, 60, 90, 180, 360])

In [ ]:
df.isnull().sum()

### Exponential Weighted Mean Features

In [ ]:
# Shift ile gecikme veriyor
# ewm ile ?s al?yor,
# en sonrada ortalamas?n? al?p transform ile d?n??t?r?yoruz.
def ewm_features(dataframe, alphas, lags):
    for alpha in alphas:
        for lag in lags:
            dataframe['deprem_ewm_alpha_' + str(alpha).replace(".", "") + "_lag_" + str(lag)] = \
                dataframe.groupby(["temizlenmis_yer"])['xm'].transform(lambda x: x.shift(lag).ewm(alpha=alpha).mean())
    return dataframe

In [ ]:
alphas = [0.95, 0.9, 0.8, 0.7, 0.5]
lags = [1, 3, 7, 30, 60, 90, 180, 360]

In [ ]:
df = ewm_features(df, alphas, lags)

In [ ]:
df.isnull().sum()

### One Hot Encoding

In [ ]:
df = pd.get_dummies(df, columns=['temizlenmis_yer'])

In [ ]:
df.shape

### Log1p D?n???m?

In [ ]:
df['xm'] = np.log1p(df["xm"].values)

### Time_based Feature and Validation Sets

In [ ]:
# 2023 3 ay? validasyon, ?ncekilerde e?itim

# 2023'nin ba??na kadar (2022'n?n sonuna kadar) train seti.
train = df.loc[(df["depremin_olus_zamani"] < "2023-01-01"), :]

# 2023'nin ilk 3'ay? validasyon seti.
val = df.loc[(df["depremin_olus_zamani"] >= "2023-01-01") & (df["depremin_olus_zamani"] < "2023-04-01"), :]

In [ ]:
# train edilirken i?imize yaramayacak olan kolonlar? ??kartal?m. Buraya daha sonra enlem ve boylam eklenecek
cols = [col for col in df.columns if col not in ['deprem kodu',"no","yer","depremin_olus_zamani","temizlenmis_yer","xm"]]

In [ ]:
# train => X_train, Y_train ....... val => X_val, Y_val
Y_train = train['xm']
X_train = train[cols]

Y_val = val['xm']
X_val = val[cols]

In [ ]:
Y_train.shape, X_train.shape, Y_val.shape, X_val.shape

## Model Developing

**Burada tekrardan Optuna kullanm?yorum.** ??nk? daha ?nce kulland?m ve sonu?lar? test ettim. O y?zden eski parametreleri tekrardan kullanarak modeli e?itiyorum.

### HistGradientBoositngRegressor

In [ ]:
optimized_params = {
    'max_iter': 774,
    'max_depth': 2,
    'min_samples_leaf': 18,
    'learning_rate': 0.0018862398822892638
}

In [ ]:
hgbr_model = HistGradientBoostingRegressor(
    max_iter=optimized_params['max_iter'],
    max_depth=optimized_params['max_depth'],
    min_samples_leaf=optimized_params['min_samples_leaf'],
    learning_rate=optimized_params['learning_rate'],
    random_state=42  # You can set the random seed for reproducibility
)


In [ ]:
hgbr_model.fit(X_train, Y_train)

# Make predictions
y_pred = hgbr_model.predict(X_val)

In [ ]:
def smape(preds, target):
    n = len(preds)
    masked_arr = ~((preds == 0) & (target == 0))
    preds, target = preds[masked_arr], target[masked_arr]
    num = np.abs(preds - target)
    denom = np.abs(preds) + np.abs(target)
    smape_val = (200 * np.sum(num / denom)) / n
    return smape_val

def hist_gradient_boosting_smape(preds, train_data):
    labels = train_data
    smape_val = smape(np.expm1(preds), np.expm1(labels))
    return 'SMAPE', smape_val, False

In [ ]:
hist_gradient_boosting_smape(y_pred, Y_val)

### LightGBMRegressor

In [ ]:
def smape(preds, target):
    n = len(preds)
    masked_arr = ~((preds == 0) & (target == 0))
    preds, target = preds[masked_arr], target[masked_arr]
    num = np.abs(preds - target)
    denom = np.abs(preds) + np.abs(target)
    smape_val = (200 * np.sum(num / denom)) / n
    return smape_val


def lgbm_smape(preds, train_data):
    labels = train_data.get_label()
    smape_val = smape(np.expm1(preds), np.expm1(labels))
    return 'SMAPE', smape_val, False

In [ ]:
lgbmr_model_parameters = {'boosting_type': 'gbdt',
             'learning_rate': 0.19986850305793402,
             'subsample': 0.9778880064060288,
             'subsample_freq': 5,
             'feature_fraction': 0.6153367760626559,
             'min_child_samples': 17,
             'min_data_in_leaf': 67,
             'objective': None,
             'num_iterations': 30000}

In [ ]:
lgbm_model = LGBMRegressor(**lgbmr_model_parameters)
lgbm_model.fit(X_train, Y_train)

In [ ]:
y_pred_val = lgbm_model.predict(X_val,
                                #num_iteration=model.best_iteration
                                )

smape(np.expm1(y_pred_val), np.expm1(Y_val))

In [ ]:
"""
lgbtrain = lgb.Dataset(data=X_train, label=Y_train, feature_name=cols)

lgbval = lgb.Dataset(data=X_val, label=Y_val, reference=lgbtrain, feature_name=cols)

lgbmr_model = lgb.train(lgbmr_model_parameters, lgbtrain,
                  valid_sets=[lgbtrain, lgbval],
                  num_boost_round =30000,
                  # early_stopping_rounds=lgb_params['early_stopping_rounds'],
                  feval=lgbm_smape,
                  # verbose_eval=1000
                  )
"""

In [ ]:
y_pred_val = lgbmr_model.predict(X_val,
                                #num_iteration=model.best_iteration
                                )

smape(np.expm1(y_pred_val), np.expm1(Y_val))

## Creating VotingRegressor 

**VotingRegression pretrained model kullanm?yor.** Bu y?zden modeli e?itmemiz gerekiyor. Fakat alternatif var:

### Creating SMAPE for VotingRegressor 

In [ ]:
def smape_scorer(y_true, y_pred):
    n = len(y_true)
    masked_arr = ~((y_pred == 0) & (y_true == 0))
    y_pred, y_true = y_pred[masked_arr], y_true[masked_arr]
    num = np.abs(y_pred - y_true)
    denom = np.abs(y_pred) + np.abs(y_true)
    smape_val = (200 * np.sum(num / denom)) / n
    return smape_val

### Creating VotingRegressor

In [ ]:
voting_regressor = VotingRegressor(estimators=[
    ('lgbmr_model', lgbmr_model),
    ('hgbr_model', hgbr_model),
])

In [ ]:
smape_scorer_make = make_scorer(smape_scorer, greater_is_better=False)

In [ ]:
scores = -cross_val_score(voting_regressor, X_train, Y_train, cv=5, scoring=smape_scorer_make,n_jobs=-1, error_score="raise")

In [ ]:
def voting_regressor_smape(models, X, y):
    # model listesi
    regressors = [(f"model_{idx}", model) for idx, model in enumerate(models)]
    
    # VotingRegressor olu?turuyorum
    voting_regressor = VotingRegressor(estimators=regressors)
    
    # Use SMAPE as the scoring metric during cross-validation
    smape_scorer_make = make_scorer(smape_scorer, greater_is_better=False)
    scores = -cross_val_score(voting_regressor, X_train, Y_train, cv=5, scoring=smape_scorer_make,n_jobs=-1, error_score="raise")

    # Calculate the mean SMAPE score across cross-validation folds
    mean_smape_score = np.mean(scores)
    return mean_smape_score


In [ ]:
voting_regressor_smape((hgbr_model,lgbm_model),X_train,Y_train)

In [ ]:
regressors = [(f"model_{idx}", model) for idx, model in enumerate((hgbr_model,lgbm_model))]
voting_regressor = VotingRegressor(estimators=regressors)

In [ ]:
voting_regressor.fit(X_train,Y_train)

**A??r? ?nemli Not:** <br>
Lighgbm'in kendi API'si Sklearn gibi model olarak d?nm?yor. Fakat Sklearn'de d?n?yor. Bu <br>
y?zden kendi API'?n? kullanmak yerin Sklearn kulladn?m. Bu y?zden Microsoft ?r?nleri be? para etmez.

### Alternative VotingRegression For Pretrained Models

VotingRegression'un Zat-? Ali'leri pretrained kabul etmedi?i i?in ?u yoluda deneyebiliriz:

In [ ]:

# preds1 = model1.predict(X_test)
# preds2 = model2.predict(X_test)

# ensemble_predictions = (preds1 + preds2) / 2  

# mse = mean_squared_error(y_test, ensemble_predictions)
# print(f"Mean Squared Error (MSE) for the ensemble: {mse}")

## Joblib

In [ ]:
joblib.dump(voting_regressor,"votingw_regressor.pkl")